# Benchmarks analysis notebook

This notebook is used to analyze the benchmarks results and to generate the files that are used in the benchmarks report.

1 - análise por dataset, samples per class, backbone, technique, 

saidas: 
combined_df_wilcoxon_freeze
combined_df_wilcoxon_ft



## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



## Extractor functions

This should be implemented by user, as it is specific to the problem. 
With executions' dataframe in hand, the user should implement/modify the following functions, in order to return the appropriate values:

- `extract_backbone`: Should return the backbone used based on information in the executions' dataframe (e.g., TS2Vec).
- `extract_tsk_pretext`: Should return the name of the pretext task used based on information in the executions' dataframe (e.g., CPC).
- `extract_d_pretext`: Should return the name of the pretext dataset used in the execution of the pretext task (e.g., KuHAR).
- `extract_tsk_target`: Should return the name of the target task (e.g., HAR, Authentication, etc).
- `extract_ft_strategy`: Should return the name of the fine-tuning strategy used on downstream task (e.g., Freeze).
- `extract_head_pred`: Should return the name of the prediction head used on downstream task (e.g., MLP).
- `extract_d_target`: Should return the name of the target dataset used in the execution of the downstream task (e.g., KuHAR).
- `extract_frac_dtarget`: Should return the fraction of the target dataset used in the execution of the downstream task (e.g., 0.1).
- `extract_metric_target`: Should return the metric used to evaluate the performance of the model on the target task (e.g., Accuracy).
- `extract_metric` should return the value of the metric used to evaluate the performance of the model on the target task, based on desired metric (e.g., 0.9).


**Note**: All extractor functions should return a list of values of same length as the number of executions.
**Note**: All functions receives the same dataframe as input (executions_df), so they can use it to extract the values. All functions should return a list of values, one for each execution (line in the dataframe).

Users should change the implementation of the following functions to match the problem at hand. Once the functions are implemented, a dataframe with the extracted values will be created. This dataframe will be used in the rest of the notebook to generate the plots and tables. Besides that, the dataframe follows the same structure as described in the H.IAAC evaluation methodology report.

In [ ]:
def extract_backbone(df):

    backbone_map = {
        'tfc_cnnpff':"CNN-PFF", 
        'rnn': "RNN",
        'resnet':"ResNet-1D",
        'tnc_resnet':"ResNet-1D",
        'cnnpff':"CNN-PFF",
       'tnc_cnnpff':"CNN-PFF", 
       'tfc_resnet':"ResNet-1D",
       'transformer':"IMU Transformer",
        'tfc_transformer':"IMU Transformer",
       'tfc_rnn': "RNN",
        'tnc_rnn': "RNN",
        'tnc_transformer':"IMU Transformer",
        'diet_rnn': "RNN",
        'diet_rnn_1': "RNN",
        'diet_rnn_2': "RNN",
        'diet_rnn_7': "RNN",
        'diet_transformer_1': "IMU Transformer",
        'diet_transformer_2': "IMU Transformer",
        'diet_transformer_7': "IMU Transformer",
        'diet_transformer': "IMU Transformer",
        'diet_cnnpff': "CNN-PFF",
        'diet_cnnpff_1': "CNN-PFF",
        'diet_cnnpff_2': "CNN-PFF",
        'diet_cnnpff_7': "CNN-PFF",
        'diet_resnet': "ResNet-1D",
        'diet_resnet_1': "ResNet-1D",
        'diet_resnet_2': "ResNet-1D",
        'diet_resnet_7': "ResNet-1D",
        'lfr_resnet_1': "ResNet-1D",
        'lfr_resnet_2': "ResNet-1D",
        'lfr_resnet_7': "ResNet-1D",
        'lfr_transformer_1': "IMU Transformer",
        'lfr_transformer_2': "IMU Transformer",
        'lfr_transformer_7': "IMU Transformer",
        'lfr_rnn_1': "RNN",
        'lfr_rnn_2': "RNN",
        'lfr_rnn_7': "RNN",
        'lfr_cnnpff_1': "CNN-PFF",
        'lfr_cnnpff_2': "CNN-PFF",
        'lfr_cnnpff_7': "CNN-PFF",
        'lfr_resnet': "ResNet-1D",
        'lfr_transformer': "IMU Transformer",
        'lfr_rnn': "RNN",
        'lfr_cnnpff': "CNN-PFF",
        'ts2vec': "TS2Vec Encoder",
        'tnc_ts2vec': "TS2Vec Encoder",
        'tfc_ts2vec': "TS2Vec Encoder",
        'diet_ts2vec': "TS2Vec Encoder",
        'diet_ts2vec_1': "TS2Vec Encoder",
        'lfr_ts2vec': "TS2Vec Encoder",
        'lfr_ts2vec_1': "TS2Vec Encoder",
        'lfr_default': "TS-TCC Encoder",
        'diet_default': "TS-TCC Encoder",
        'tfc_default': 'TFCCNNEncoder',
        'tfc_harcnn': 'TS-TCC Encoder',
        'tnc_harcnn': 'TS-TCC Encoder',
        'harcnn': 'TS-TCC Encoder',
        'resnetse5': "ResNet-SE-5",
        'tfc_resnetse5': "ResNet-SE-5",
        'tnc_resnetse5': "ResNet-SE-5",
        'diet_resnetse5': "ResNet-SE-5",
        'lfr_resnetse5': "ResNet-SE-5",

    }


    return df["model/name"].map(backbone_map)



def extract_tsk_pretext(df):
    tsk_pretext_map = {
        'tfc_cnnpff':"TFC", 
        'rnn': "Supervised",
        'resnet':"Supervised",
        'tnc_resnet':"TNC",
        'cnnpff':"Supervised",
       'tnc_cnnpff':"TNC", 
       'tfc_resnet':"TFC",
       'transformer':"Supervised",
        'tfc_transformer':"TFC",
       'tfc_rnn': "TFC",
        'tnc_rnn': "TNC",
        'tnc_transformer':"TNC",
        'diet_rnn': "Diet",
        'diet_transformer': "Diet",
        'diet_cnnpff': "Diet",
        'diet_resnet': "Diet",
        'diet_rnn_1': "Diet",
        'diet_transformer_1': "Diet",
        'diet_cnnpff_1': "Diet",
        'diet_resnet_1': "Diet",
        'lfr_rnn': "LFR",
        'lfr_transformer': "LFR",
        'lfr_cnnpff': "LFR",
        'lfr_resnet': "LFR",
        'lfr_rnn_1': "LFR",
        'lfr_transformer_1': "LFR",
        'lfr_cnnpff_1': "LFR",
        'lfr_resnet_1': "LFR",
        'lfr_rnn_2': "LFR",
        'lfr_transformer_2': "LFR",
        'lfr_cnnpff_2': "LFR",
        'lfr_resnet_2': "LFR",
        'lfr_rnn_7': "LFR",
        'lfr_transformer_7': "LFR",
        'lfr_cnnpff_7': "LFR",
        'lfr_resnet_7': "LFR",
        'diet_rnn_2': "Diet",
        'diet_transformer_2': "Diet",
        'diet_cnnpff_2': "Diet",
        'diet_resnet_2': "Diet",
        'diet_rnn_7': "Diet",
        'diet_transformer_7': "Diet",
        'diet_cnnpff_7': "Diet",
        'diet_resnet_7': "Diet",
        'ts2vec': "Supervised",
        'tnc_ts2vec': "TNC",
        'tfc_ts2vec': "TFC",
        'diet_ts2vec': "Diet",
        'diet_ts2vec_1': "Diet",
        'lfr_ts2vec': "LFR",
        'lfr_ts2vec_1': "LFR",
        'lfr_default': "LFR",
        'diet_default': "Diet",
        'tfc_default': 'TFC',
        'tfc_harcnn': 'TFC',
        'tnc_harcnn': 'TNC',
        'harcnn': 'Supervised',
        'resnetse5': "Supervised",
        'tfc_resnetse5': "TFC",
        'tnc_resnetse5': "TNC",
        'diet_resnetse5': "Diet",
        'lfr_resnetse5': "LFR",

        
    }
    return df["model/name"].map(tsk_pretext_map)


def extract_d_pretext(df):
    d_pretext_map = {
        "kuhar": "KH",
        "motionsense": "MS",
        "rw_thigh": "RW-Thigh",
        "rw_waist": "RW-Waist",
        "uci": "UCI",
        "wisdm": "WISDM",
        "hapt": "HAPT",
        "recodgait": "RecodGait",
    }
    return df["data/dataset"].map(d_pretext_map)


def extract_ft_strategy(df):
    def extract_ft_by_name(row):
        if "freeze" in row["model/override_id"]:
            return "Freeze"
        else:
            return "Full Finetune"
    
    return df.apply(extract_ft_by_name, axis=1)

def extract_head_pred(df):
    return ["MLP"] * len(df)


def extract_tsk_target(df):
    tsk_map = {
        "har": "HAR",
    }
    return df["pipeline/task"].map(tsk_map)


def extract_d_target(df):
    d_target_map = {
        "kuhar": "KH",
        "motionsense": "MS",
        "rw_thigh": "RW-Thigh",
        "rw_waist": "RW-Waist",
        "uci": "UCI",
        "wisdm": "WISDM",
        "hapt": "HAPT",
    }
    return df["data/dataset"].map(d_target_map)



def extract_frac_dtarget(df):
    def get_mix_percentage(row):
        frac_dtarget_map = {
            "multimodal_samples_001": 1,
            "multimodal_samples_001_2": 1,
            "multimodal_samples_001_3": 1,
            "multimodal_samples_005": 5,
            "multimodal_samples_005_2": 5,
            "multimodal_samples_005_3": 5,
            "multimodal_samples_010": 10,
            "multimodal_samples_010_2": 10,
            "multimodal_samples_010_3": 10,
            "multimodal_samples_025": 25,
            "multimodal_samples_025_2": 25,
            "multimodal_samples_025_3": 25,
            "multimodal_samples_050": 50,
            "multimodal_samples_050_2": 50,
            "multimodal_samples_050_3": 50,
            "multimodal_samples_100": 100,
            "multimodal_samples_100_2": 100,
            "multimodal_samples_100_3": 100,
            "multimodal_samples_200": 200,
            "multimodal_samples_200_2": 200,
            "multimodal_samples_200_3": 200,
            "multimodal_perc_100": 1000,
            "multimodal_perc_100_2": 1000,
            "multimodal_perc_100_3": 1000,
            }
        if not pd.isna(row["backbone/load_from_uid"] ):
            try:
                row = df.loc[df["execution/uid"] == row["backbone/load_from_uid"]].iloc[0]

                frac = row["data/override_id"]
                return frac_dtarget_map[frac]
            except Exception as e:
                print(f"Invalid load_from_uid: {row['backbone/load_from_uid']}:\n {e}")
                return 0
        
    return df.apply(get_mix_percentage, axis=1)


In [ ]:
def aggregate_results(df: pd.DataFrame) -> pd.DataFrame:
    """Group the dataframe by all columns except the metric and aggregate the
    metric values using the mean and standard deviation.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be aggregated

    Returns
    -------
    _type_
        _description_
    """
    agg_df = []
    every_column_expect_metric = [col for col in df.columns if col != "metric"]

    for _, grouped_df in df.groupby(every_column_expect_metric):
        grouped_df = grouped_df.copy()
        mean = grouped_df["metric"].mean()
        stdev = grouped_df["metric"].std()
        if pd.isna(stdev):
            stdev = 0.0
        grouped_df["metric"] = mean
        grouped_df["metric_stdev"] = stdev
        grouped_df["run_count"] = len(grouped_df)

        # print(f"{group_name}, with values: {grouped_df['metric'].values} -- {mean:.2f} ± {stdev:.2f}")
        single_line = grouped_df.iloc[0]
        agg_df.append(single_line)

    df = pd.DataFrame(agg_df).reset_index(drop=True)
    return df


def parse_dataframe(
    df: pd.DataFrame,
    filter_nan_metric: bool = True,
    aggregate_runs: bool = False,
) -> pd.DataFrame:
    """Parse the dataframe to extract the relevant columns and values for the
    experiment. If `filter_nan_metric` is True, the rows with NaN metric values
    are removed. If `aggregate_runs` is True, the results are aggregated by
    grouping the dataframe by all columns except the metric and calculating the
    mean and standard deviation of the metric values.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be parsed
    filter_nan_metric : bool, optional
        If True the rows with NaN metric values are removed, by default True
    aggregate_runs : bool, optional
        If True the results are aggregated, by default False

    Returns
    -------
    pd.DataFrame
        The parsed dataframe
    """
    d = {
        # "uid": pd.Series(df["execution/uid"], dtype="object"),
        "backbone": pd.Series(extract_backbone(df), dtype="object"),
        "tsk_pretext": pd.Series(extract_tsk_pretext(df), dtype="object"),
        "d_pretext": pd.Series(extract_d_pretext(df), dtype="object"),
        "head_pred": pd.Series(extract_head_pred(df), dtype="object"),
        "ft_strategy": pd.Series(extract_ft_strategy(df), dtype="object"),
        "tsk_target": pd.Series(extract_tsk_target(df), dtype="object"),
        "d_target": pd.Series(extract_d_target(df), dtype="object"),
        "frac_dtarget": pd.Series(extract_frac_dtarget(df), dtype="float"),
        "metric_target": pd.Series(extract_metric_target(df), dtype="object"),
        "metric": pd.Series(extract_metric(df), dtype="float"),
    }

    df = pd.DataFrame(d)

    if filter_nan_metric:
        df = df.dropna(subset=["metric"])

    if aggregate_runs:
        df = aggregate_results(df)
    return df

In [ ]:
df = pd.read_csv(summarized_executions_path)
df

In [ ]:
# start_idx = 7500
# end_idx = 8500  # exclusive

# # Step 1: Split the DataFrame into three parts
# before = df.iloc[:start_idx]
# middle = df.iloc[start_idx:end_idx]
# after = df.iloc[end_idx:]

# # Step 2: Filter out rows containing 'tnc_resnet' in the middle part
# # You can specify the column or check across all columns (assuming all string columns)
# # Here's how to check all columns for 'tnc_resnet' string:
# mask = ~middle.astype(str).apply(lambda row: row.str.contains('tnc_resnet')).any(axis=1)
# middle_filtered = middle[mask]

# filter = False

# # Step 3: Concatenate back
# if filter:
#     df = pd.concat([before, middle_filtered, after], ignore_index=True)

# df

In [ ]:
df['model/name'].unique()

In [ ]:
df

In [ ]:
df[df['metric/classification/balanced_accuracy'] == 0]

In [ ]:
# df['metric/classification/balanced_accuracy'] = df['metric/classification/balanced_accuracy'].apply(
#     lambda x: float(0.01) if x == 0 else x
# )


In [ ]:
df[df['metric/classification/balanced_accuracy'] == 0]

In [ ]:
df

In [ ]:
df = parse_dataframe(df, filter_nan_metric=True, aggregate_runs=False)
df

In [ ]:
df["frac_dtarget"] = df["frac_dtarget"].replace("0.0", "1000.0")

In [ ]:
df[df['metric'] == 0.01]

In [ ]:
df[df['metric'] == 0.1]

In [ ]:
df

In [ ]:
df["frac_dtarget"] = df["frac_dtarget"].astype(str)
# df = df[~df['frac_dtarget'].str.contains('100')]
df

In [ ]:
# df = df[~df['frac_dtarget'].str.contains('50')]
df

In [ ]:
df = df[~df['d_pretext'].str.contains('HAPT')]

In [ ]:
df

In [ ]:
# df = df[~df['tsk_pretext'].str.contains('LFR')]

In [ ]:
df_lfr = df[df['tsk_pretext'].str.contains('LFR')]
# df_lfr = df_lfr[df_lfr['backbone'].str.contains('TS')]
df_lfr

In [ ]:
df_tfc = df[df['tsk_pretext'].str.contains('TFC')]
df_tfc

In [ ]:
df

In [ ]:
df.info()

In [ ]:
from tabulate import tabulate  # optional, for prettier markdown

columns = ['tsk_pretext', 'backbone', 'd_target', 'frac_dtarget', 'ft_strategy']

# Prepare the rows
rows = []
for col in columns:
    counts = df[col].value_counts()
    summary = ", ".join([f"{k} ({v})" for k, v in counts.items()])
    rows.append((col, summary))

# Print Markdown Table
print(f'number of experiments: {len(df)}')
print("## 📊 Markdown Summary Table\n")
print(tabulate(rows, headers=["Variable", "Value Counts"], tablefmt="github"))

# Build LaTeX Table
latex_table = "\\begin{table}[ht]\n\\centering\n"
latex_table += "\\begin{tabular}{ll}\n"
latex_table += "\\toprule\n"
latex_table += "Variable & Value Counts \\\\\n"
latex_table += "\\midrule\n"
for var, vals in rows:
    latex_table += f"{var} & {vals} \\\\\n"
latex_table += "\\bottomrule\n"
latex_table += "\\end{tabular}\n"
latex_table += "\\caption{Summary of experimental settings used across different configurations.}\n"
latex_table += "\\label{tab:exp-summary}\n"
latex_table += "\\end{table}"

print("\n\n## 📄 LaTeX Table\n")
print(latex_table)


In [ ]:
# convert fracdtarget 0.00 to 1000
df["frac_dtarget"] = df["frac_dtarget"].replace("0.0", "1000.0")

In [ ]:


df = df[df['backbone']!="ResNet-1D"]

In [ ]:
from tabulate import tabulate  # optional, for prettier markdown

columns = ['tsk_pretext', 'backbone', 'd_target', 'frac_dtarget', 'ft_strategy']

# Prepare the rows
rows = []
for col in columns:
    counts = df[col].value_counts()
    summary = ", ".join([f"{k} ({v})" for k, v in counts.items()])
    rows.append((col, summary))

# Print Markdown Table
print(f'number of experiments: {len(df)}')
print("## 📊 Markdown Summary Table\n")
print(tabulate(rows, headers=["Variable", "Value Counts"], tablefmt="github"))

# Build LaTeX Table
latex_table = "\\begin{table}[ht]\n\\centering\n"
latex_table += "\\begin{tabular}{ll}\n"
latex_table += "\\toprule\n"
latex_table += "Variable & Value Counts \\\\\n"
latex_table += "\\midrule\n"
for var, vals in rows:
    latex_table += f"{var} & {vals} \\\\\n"
latex_table += "\\bottomrule\n"
latex_table += "\\end{tabular}\n"
latex_table += "\\caption{Summary of experimental settings used across different configurations.}\n"
latex_table += "\\label{tab:exp-summary}\n"
latex_table += "\\end{table}"

print("\n\n## 📄 LaTeX Table\n")
print(latex_table)


In [ ]:
df.to_csv(f'clean_{filename}.csv', index=False)

In [ ]:
base_df = df.copy()

In [ ]:
# tfc cnn ts2vec across datasets and fractions
import numpy as np
# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Freeze"],
                "select_tsk_pretext": ["TFC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TFC",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# supervised cnn ts2vec across datasets and fractions

# with ts2vec partial

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial


# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# # freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Freeze"],
                "select_tsk_pretext": ["TNC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Freeze"],
                "select_tsk_pretext": ["Diet"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Diet",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Freeze"],
                "select_tsk_pretext": ["LFR"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "LFR",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# technique_summary_supervised

In [ ]:

combined_df_freeze = pd.concat([
    # technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_diet,
    technique_summary_tnc
])


combined_df_freeze = combined_df_freeze.drop_duplicates(subset=['Backbone','Technique'])
combined_df_freeze

combined_df_freeze['Data Percentage'] = combined_df_freeze['Backbone'].str.split(r'\s+\+\s+').str[2].str.replace('.0', '').astype(int)
combined_df_freeze['Dataset Name'] = combined_df_freeze['Backbone'].str.split(r'\s+\+\s+').str[1]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df_freeze['Backbone'] = combined_df_freeze['Backbone'].str.split(r'\s+\+\s+').str[0]
combined_df_freeze

combined_df_freeze['ft_strategy'] = 'Freeze'
combined_df_freeze
combined_df_freeze.to_csv(f'combined_df_wilcoxon_freeze_{filename}.csv', index=False)


## full finetune

In [ ]:
# tfc cnn ts2vec across datasets and fractions
import numpy as np
# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["TFC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TFC",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# # supervised cnn ts2vec across datasets and fractions

# # with ts2vec partial

# # freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["Supervised"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Supervised",
]

# Generate all tables
combined_df, technique_summary_supervised, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_supervised)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# # freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["TNC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["Diet"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Diet",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



In [ ]:
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["LFR"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "LFR",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:

combined_df_ft = pd.concat([
    technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_diet,
    technique_summary_tnc
])


combined_df_ft = combined_df_ft.drop_duplicates(subset=['Backbone','Technique'])
combined_df_ft

combined_df_ft['Data Percentage'] = combined_df_ft['Backbone'].str.split(r'\s+\+\s+').str[2].str.replace('.0', '').astype(int)
combined_df_ft['Dataset Name'] = combined_df_ft['Backbone'].str.split(r'\s+\+\s+').str[1]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df_ft['Backbone'] = combined_df_ft['Backbone'].str.split(r'\s+\+\s+').str[0]
combined_df_ft

combined_df_ft['ft_strategy'] = 'Full Finetuning'
combined_df_ft
combined_df_ft.to_csv(f'combined_df_wilcoxon_ft_{filename}.csv', index=False)
